# MMLogQA Construction & Analysis Pipeline


## Setup

In [1]:
import re, ast
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

VALID_IMG_PATTERN = re.compile(r'^https://i\.sstatic\.net/[A-Za-z0-9]+\.(jpg|png)$', re.I)

## Utility Functions

### List Parsing

In [2]:
def parse_list(val):
    if isinstance(val, list): return val
    try: return ast.literal_eval(val) if pd.notna(val) and val != '' else []
    except: return []

### Best Answer Selection

In [3]:
def get_best_answer(row):
    if pd.notna(row['selected_answer']): return row['selected_answer']
    ans_list = parse_list(row['answers'])
    valid = [a for a in ans_list if int(str(a.get('score') or 0).strip()) >= 2]
    return max(valid, key=lambda x: int(str(x.get('score') or 0).strip()))['body'].strip() if valid else None

### Score Reliability Filter

In [4]:
def score_filter(row, threshold=2):
    ans_list = parse_list(row['answers'])
    sel_body = str(row.get('selected_answer', '')).strip()
    return any(str(a.get('body', '')).strip() == sel_body and int(str(a.get('score') or 0).strip()) > threshold for a in ans_list)

### Image URL Filter

In [5]:
def url_filter(imgs):
    return [u for u in parse_list(imgs) if VALID_IMG_PATTERN.match(u.strip())]

### Stratified Splitting

In [6]:
def stratified_split(df, seed=42):
    df = df.copy()
    df['tag'] = df['tags'].apply(lambda x: (parse_list(x) or ['unknown'])[0])
    counts = df['tag'].value_counts()
    df['strat'] = df['tag'].apply(lambda t: t if counts[t] >= 3 else 'other')
    tr_val, te = train_test_split(df, test_size=0.2, stratify=df['strat'], random_state=seed)
    tv_counts = tr_val['strat'].value_counts()
    tr_val['strat'] = tr_val['strat'].apply(lambda t: t if tv_counts[t] >= 2 else 'other')
    tr, val = train_test_split(tr_val, test_size=0.25, stratify=tr_val['strat'], random_state=seed)
    return tr.drop(columns=['tag','strat']), val.drop(columns=['tag','strat']), te.drop(columns=['tag','strat'])

---
# Cloud Dataset

### Loading & Inspection

In [7]:
c_raw = pd.read_csv('../data/raw/cloud_multimodal_qa_raw.csv')
print(f'Raw Shape: {c_raw.shape}')

Raw Shape: (45796, 11)


In [8]:
c_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45796 entries, 0 to 45795
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            45796 non-null  object
 1   body             45796 non-null  object
 2   tags             45796 non-null  object
 3   link             45796 non-null  object
 4   score            45796 non-null  int64 
 5   creation_date    45796 non-null  object
 6   answer_count     45796 non-null  int64 
 7   comments         45796 non-null  object
 8   answers          45796 non-null  object
 9   selected_answer  45796 non-null  object
 10  images           45796 non-null  object
dtypes: int64(2), object(9)
memory usage: 3.8+ MB


In [9]:
c_raw[['answers', 'images']].head(2)

,answers,images
0,[{'body': '\n \nWhen I ran belo...,"['https://i.imgur.com/NwVqGBF.png', 'https://i..."
1,[{'body': '\nTo have one parameter depend on a...,"['https://i.sstatic.net/y3Iiq.png', 'https://i..."


### Preprocessing

In [10]:
c_df = c_raw.drop_duplicates('title').reset_index(drop=True)
print(f'After Dedup: {c_df.shape}')

After Dedup: (42647, 11)


In [11]:
c_df['selected_answer'] = c_df.apply(get_best_answer, axis=1)
c_df = c_df[c_df['selected_answer'].notnull()].reset_index(drop=True)
print(f'After Answer Selection: {c_df.shape}')

After Answer Selection: (42647, 11)


### Filtering

In [12]:
c_df = c_df[c_df['images'].apply(lambda x: len(parse_list(x)) > 0)].reset_index(drop=True)
print(f'With Images: {c_df.shape}')

With Images: (42647, 11)


In [13]:
c_df = c_df[c_df.apply(lambda r: score_filter(r, 2), axis=1)].reset_index(drop=True)
print(f'Score > 2: {c_df.shape}')

ValueError: invalid literal for int() with base 10: 'None'

In [14]:
c_df['images'] = c_df['images'].apply(url_filter)
c_df = c_df[c_df['images'].apply(len) > 0].reset_index(drop=True)
print(f'Valid URLs: {c_df.shape}')

Valid URLs: (40351, 11)


### Splitting (60/20/20)

In [15]:
c_tr, c_val, c_te = stratified_split(c_df)
for n, d in zip(['Train', 'Val', 'Test'], [c_tr, c_val, c_te]): print(f'{n}: {d.shape}')

Train: (24210, 11)
Val: (8070, 11)
Test: (8071, 11)


---
# Device Dataset

### Loading & Inspection

In [16]:
d_raw = pd.read_csv('../data/raw/device_multimodal_qa_raw.csv')
print(f'Raw Shape: {d_raw.shape}')

Raw Shape: (71359, 11)


In [17]:
d_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71359 entries, 0 to 71358
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   title            71359 non-null  object
 1   body             71359 non-null  object
 2   tags             71359 non-null  object
 3   link             71359 non-null  object
 4   score            71359 non-null  int64 
 5   creation_date    71359 non-null  object
 6   answer_count     71359 non-null  int64 
 7   comments         71359 non-null  object
 8   answers          71359 non-null  object
 9   selected_answer  71359 non-null  object
 10  images           71359 non-null  object
dtypes: int64(2), object(9)
memory usage: 6.0+ MB


In [18]:
d_raw[['answers', 'images']].head(2)

,answers,images
0,[{'body': '\nUse keywords for your search; oth...,['https://i.sstatic.net/0Ukoc.png']
1,[{'body': '\nYou say you are trying to attack ...,"['https://i.sstatic.net/mSEof.jpg', 'https://i..."


### Preprocessing

In [19]:
d_df = d_raw.drop_duplicates('title').reset_index(drop=True)
print(f'After Dedup: {d_df.shape}')

After Dedup: (7317, 11)


In [20]:
d_df['selected_answer'] = d_df.apply(get_best_answer, axis=1)
d_df = d_df[d_df['selected_answer'].notnull()].reset_index(drop=True)
print(f'After Answer Selection: {d_df.shape}')

After Answer Selection: (7317, 11)


### Filtering

In [21]:
d_df = d_df[d_df['images'].apply(lambda x: len(parse_list(x)) > 0)].reset_index(drop=True)
print(f'With Images: {d_df.shape}')

With Images: (7317, 11)


In [22]:
d_df = d_df[d_df.apply(lambda r: score_filter(r, 0), axis=1)].reset_index(drop=True)
print(f'Score > 0: {d_df.shape}')

Score > 0: (6229, 11)


In [23]:
d_df['images'] = d_df['images'].apply(url_filter)
d_df = d_df[d_df['images'].apply(len) > 0].reset_index(drop=True)
print(f'Valid URLs: {d_df.shape}')

Valid URLs: (5993, 11)


### Splitting (60/20/20)

In [24]:
d_tr, d_val, d_te = stratified_split(d_df)
for n, d in zip(['Train', 'Val', 'Test'], [d_tr, d_val, d_te]): print(f'{n}: {d.shape}')

Train: (3595, 11)
Val: (1199, 11)
Test: (1199, 11)


---
# Final Summary

In [25]:
summary = pd.DataFrame({
    'Dataset': ['Cloud', 'Device'],
    'Raw': [len(c_raw), len(d_raw)],
    'Filtered': [len(c_df), len(d_df)],
    'Train': [len(c_tr), len(d_tr)],
    'Val': [len(c_val), len(d_val)],
    'Test': [len(c_te), len(d_te)]
})
summary

,Dataset,Raw,Filtered,Train,Val,Test
0,Cloud,45796,40351,24210,8070,8071
1,Device,71359,5993,3595,1199,1199
